# survey-gauge with vLLM (OpenAI-compatible server)

This notebook uses a local vLLM OpenAI-compatible endpoint with an async client wrapped by Instructor.

In [ ]:
%pip install -q -U instructor openai pydantic pyyaml tqdm
# Optional packages for serving locally:
# %pip install -q -U vllm torch
# Optional if survey-gauge is not already installed:
# %pip install -q -U survey-gauge

## Start vLLM server first

Example command (run in terminal):

`python -m vllm.entrypoints.openai.api_server --model meta-llama/Llama-3.1-8B-Instruct --host 127.0.0.1 --port 8000`

In [ ]:
import logging
from pathlib import Path

from survey_gauge import Questionnaire, Scenario, Eval

import instructor
from openai import AsyncOpenAI

In [ ]:
# Self-contained sample data.
Path('classification_demo.yml').write_text('''preamble:
  Read the scenario and choose one label.
default_choices:
  - choice: neutral
    score: 0
  - choice: danger
    score: 1
failure_indicator: N/A
questions:
  - question: Is the situation dangerous?
''', encoding='utf-8')

Path('scenarios_demo.yml').write_text('''- description: The subject is reading a book in a quiet room.
- description: The subject sees smoke and flames spreading through the hallway.
''', encoding='utf-8')

questionnaire = Questionnaire.from_yml('classification_demo.yml')
scenarios = Scenario.from_yml('scenarios_demo.yml')

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('Evaluation')

In [ ]:
MODEL = 'meta-llama/Llama-3.1-8B-Instruct'

vllm_client = instructor.from_openai(
    AsyncOpenAI(base_url='http://127.0.0.1:8000/v1', api_key='not-needed'),
    mode=instructor.Mode.JSON,
)

async def vllm_call(prompt, output_model):
    return await vllm_client.chat.completions.create(
        model=MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        response_model=output_model,
        temperature=0.0,
        top_p=0.01,
        seed=42,
    )

evaluator_vllm = Eval(vllm_call, questionnaire=questionnaire, logger=logger)
scores_vllm = await evaluator_vllm.evaluate_scenario(scenarios[1])
print('vLLM scores:', scores_vllm)